# Copernicus DEM — download & crop

Fetch a 1-degree GLO-30 elevation tile from the AWS Open Data mirror, cropped to a
small AOI, and inspect the resulting WGS84 COG with `pyramids`.

## Setup

A couple of imports: `tempfile` / `Path` give us a scratch directory for the
downloaded tile, and `EarthLens` is the unified entry point.

In [ ]:
import tempfile
from pathlib import Path

from earthlens.core import EarthLens

## Build the request

The Copernicus DEM lives on a public AWS bucket, so we use the `amazon-s3` backend.
We point it at the `copernicus-dem` dataset, a small AOI off the Gulf of Guinea, and
a scratch output directory.

In [ ]:
out = Path(tempfile.mkdtemp(prefix='dem_'))
src = EarthLens(
    data_source="amazon-s3",
    start='2021-01-01',
    end='2021-01-01',
    aoi=[6.40, 0.40, 6.45, 0.45],
    dataset='copernicus-dem',
    path=str(out),
)

## Download the tile

`download()` fetches the cropped tile to `out` and returns the list of written paths.

In [ ]:
paths = src.download(progress_bar=False)
paths

## Inspect the result

`pyramids` reads the written COG so we can confirm its CRS, bounding box, and shape
match the requested AOI.

In [ ]:
from pyramids.dataset import Dataset

ds = Dataset.read_file(paths[0])
print('CRS EPSG:', ds.epsg)
print('bbox:', [round(x, 3) for x in ds.bbox])
print('shape:', ds.shape)